In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torchaudio")

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torchaudio import datasets, transforms
import random

In [3]:
COMMANDS_10 = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']
LABELS = COMMANDS_10 + ['unknown', 'silence']

labelToIdx = {l: i for i, l in enumerate(LABELS)}
NUM_CLASSES = len(LABELS)  # 12

In [4]:
N_MELS = 16

In [5]:
mel_spec = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=512,
    hop_length=160,  
    win_length=480,  # 480 (30ms)
    n_mels=N_MELS,       # 16 bins (FIX) + 16 Delta + 16 Delta-Delta
    f_min=0.0,
    f_max=8000.0,
)

In [6]:
TARGET_LENGTH = 16000

def collate_fn(batch, training=False):
    waveforms, labels = [], []

    for waveform, sample_rate, label, *_ in batch:
        # Pad/trim
        if waveform.shape[-1] < TARGET_LENGTH:
            waveform = F.pad(waveform, (0, TARGET_LENGTH - waveform.shape[-1]))
        else:
            waveform = waveform[..., :TARGET_LENGTH]

        # Time shift augmentation
        if training:
            shift = random.randint(-800, 800)
            if shift > 0:
                waveform = F.pad(waveform[..., :-shift], (shift, 0))
            elif shift < 0:
                waveform = F.pad(waveform[..., -shift:], (0, -shift))
            noise_amp = random.uniform(0.0, 0.005)
            waveform = waveform + noise_amp * torch.randn_like(waveform)

        if label in COMMANDS_10:
            mapped_label = labelToIdx[label]
        else:
            mapped_label = labelToIdx['unknown']

        waveforms.append(waveform)
        labels.append(mapped_label)

    # Add silence samples (~10%)
    num_silence = int(0.1 * len(waveforms))
    for _ in range(num_silence):
        noise_amp = random.uniform(0.0, 0.002)
        silent = torch.randn(1, TARGET_LENGTH) * noise_amp
        waveforms.append(silent)
        labels.append(labelToIdx['silence'])

    return torch.stack(waveforms), torch.tensor(labels)

In [7]:
train_collate = lambda batch: collate_fn(batch, training=True)
test_collate  = lambda batch: collate_fn(batch, training=False)

In [8]:
SC_train = datasets.SPEECHCOMMANDS(root='data', subset='training', download=True, url='speech_commands_v0.02')
SC_val   = datasets.SPEECHCOMMANDS(root='data', subset='validation', download=True, url='speech_commands_v0.02')
SC_test  = datasets.SPEECHCOMMANDS(root='data', subset='testing',  download=True, url='speech_commands_v0.02')

In [9]:
train_loader = torch.utils.data.DataLoader(SC_train, batch_size=64, num_workers=4, pin_memory=True, persistent_workers=True, shuffle=True, collate_fn=train_collate)
val_loader   = torch.utils.data.DataLoader(SC_val, batch_size=1024, num_workers=4, pin_memory=True, persistent_workers=True, shuffle=False, collate_fn=test_collate)
test_loader  = torch.utils.data.DataLoader(SC_test, batch_size=1024, num_workers=4, pin_memory=True, persistent_workers=True, shuffle=False, collate_fn=test_collate)

In [10]:
class KeywordGRU(nn.Module):
    def __init__(self, n_mels=N_MELS, hidden_size=64, num_layers=2, num_classes=NUM_CLASSES): # Change to hidden size 64
        super().__init__()
        self.mel = mel_spec
        self.db = torchaudio.transforms.AmplitudeToDB()

        self.compute_deltas = torchaudio.transforms.ComputeDeltas()
        
        # self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=8)
        # self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=12)

        self.gru = nn.GRU(
            input_size=n_mels*3,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2 # Dropout potentially not needed
        )

        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.mel(x)        # (batch, 1, n_mels, time)
        x = self.db(x)         # log-mel

        deltas = self.compute_deltas(x)
        ddeltas = self.compute_deltas(deltas)
        x = torch.cat([x, deltas, ddeltas], dim=2) # (batch, 1, n_mels*3, time)
        
        std = x.std(dim=-1, keepdim=True).clamp(min=0.1)
        x = (x - x.mean(dim=-1, keepdim=True)) / std
        
        # if self.training:
        #    x = self.freq_mask(x)
        #    x = self.time_mask(x)
        x = x.squeeze(1)       # (batch, n_mels, time)
        x = x.permute(0, 2, 1) # (batch, time, n_mels)

        out, h_n = self.gru(x)           # (num_layers, batch, hidden)
        last_hidden = h_n[-1]            # (batch, hidden) last layer's final state
        return self.classifier(last_hidden)

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KeywordGRU().to(device)
EPOCHS = 100

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS + 1,
    pct_start=0.3
) # Cosine scheduler (CosineAnnealingLR etc.)

class_weights = torch.ones(NUM_CLASSES, device=device)
class_weights[labelToIdx['unknown']] = 1.20
class_weights[labelToIdx['silence']] = 1.10

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05 # Could be off
)

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total model parameters: {total_params:,}")

Device: cuda
GPU Name: NVIDIA H200 MIG 2g.35gb
GPU Memory: 34.90 GB
Total model parameters: 47,628


In [17]:
def train(epoch=None):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Could be off
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        preds = output.argmax(dim=1)
        correct += (preds == target).sum().item()
        total += target.size(0)

        if batch_idx % 500 == 0 and epoch is not None:
            print(f"Epoch {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}")

    avg_loss = total_loss / len(train_loader)
    accuracy = 100. * correct / total

    if epoch is not None:
        print(f"Epoch {epoch} - Avg Loss: {avg_loss:.6f} | Train Acc: {accuracy:.2f}%")

    return accuracy

In [18]:
def evaluate(loader, split_name="Val"):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            total_loss += loss.item() * target.size(0)
            preds = output.argmax(dim=1)
            correct += (preds == target).sum().item()
            total += target.size(0)

    avg_loss = total_loss / total
    acc = 100.0 * correct / total
    print(f"{split_name}: loss={avg_loss:.4f}, acc={correct}/{total} ({acc:.2f}%)")
    return acc, avg_loss

In [ ]:
import matplotlib.pyplot as plt

best_val = -1.0
bad_epochs = 0
patience = 8

train_acc_history = []
val_acc_history = []

for epoch in range(1, EPOCHS + 1):
    train_acc = train(epoch)
    val_acc, val_loss = evaluate(val_loader, "Val")

    train_acc_history.append(train_acc)
    val_acc_history.append(val_acc)
    
    if val_acc > best_val:
        best_val = val_acc
        bad_epochs = 0
        torch.save(model.state_dict(), "best_keyword_gru.pt")
        print(f"saved best: {best_val:.2f}%")

Epoch 1 [0/84843] Loss: 2.427177
Epoch 1 [35000/84843] Loss: 1.739256
Epoch 1 [70000/84843] Loss: 1.491594
Epoch 1 - Avg Loss: 1.677463 | Train Acc: 57.65%
Val: loss=1.5247, acc=6768/10975 (61.67%)
saved best: 61.67%
Epoch 2 [0/84843] Loss: 1.577261
Epoch 2 [35000/84843] Loss: 1.419789
Epoch 2 [70000/84843] Loss: 1.583131
Epoch 2 - Avg Loss: 1.388323 | Train Acc: 65.89%
Val: loss=1.3802, acc=7262/10975 (66.17%)
saved best: 66.17%
Epoch 3 [0/84843] Loss: 1.426442
Epoch 3 [35000/84843] Loss: 1.505153
Epoch 3 [70000/84843] Loss: 1.114781
Epoch 3 - Avg Loss: 1.301117 | Train Acc: 66.80%
Val: loss=1.2627, acc=7382/10975 (67.26%)
saved best: 67.26%
Epoch 4 [0/84843] Loss: 1.178144
Epoch 4 [35000/84843] Loss: 1.196986
Epoch 4 [70000/84843] Loss: 1.552866
Epoch 4 - Avg Loss: 1.139883 | Train Acc: 68.61%
Val: loss=1.1455, acc=7442/10975 (67.81%)
saved best: 67.81%
Epoch 5 [0/84843] Loss: 1.268407
Epoch 5 [35000/84843] Loss: 0.920165
Epoch 5 [70000/84843] Loss: 1.063799
Epoch 5 - Avg Loss: 1.024

In [ ]:
model.load_state_dict(torch.load("best_keyword_gru.pt", map_location=device))
test_acc, test_loss = evaluate(test_loader, "Test")

epochs_ran = range(1, len(val_acc_history) + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, train_acc_history, marker='^', linewidth=2, label='Training Accuracy')
plt.plot(epochs_ran, val_acc_history, marker='o', linewidth=2, label='Validation Accuracy')

plt.axhline(y=[test_acc], color='r', linestyle='--', label='Final Test Accuracy')

plt.annotate(f"Test: {test_acc:.2f}%", (len(val_acc_history), test_acc), textcoords='offset points', xytext=(10, 6))
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training and Validation Accuracy with Final Test Marker')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()